# EEE 475/575 Medical Image Reconstruction & Processing — Homework #1

**GenAI Usage Disclosure:** GitHub Copilot (Claude Opus 4.6) was used to assist with code generation and structuring of the notebook.

## Preparation 1–5: Imports, Centered FFT Functions, Display Helpers, and Data Loading

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from skimage.metrics import peak_signal_noise_ratio as compute_psnr
from skimage.metrics import structural_similarity as compute_ssim
from skimage.io import imread
from skimage.color import rgb2gray

%matplotlib inline
plt.rcParams['figure.figsize'] = [8, 6]
plt.rcParams['figure.dpi'] = 100

### Preparation 1: Centered FFT Functions
These functions perform centered FFT/IFFT so that the origin (DC component) is at the center of the array, matching the convention used in medical image reconstruction.

In [ ]:
# --- Preparation 1: Centered FFT functions ---

def fft2c(im):
    """Centered 2D FFT: shifts origin to center before/after FFT."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(im)))

def ifft2c(d):
    """Centered 2D inverse FFT: shifts origin to center before/after IFFT."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(d)))

def fftc(im):
    """Centered 1D FFT."""
    return np.fft.fftshift(np.fft.fft(np.fft.ifftshift(im)))

def ifftc(d):
    """Centered 1D inverse FFT."""
    return np.fft.fftshift(np.fft.ifft(np.fft.ifftshift(d)))

# Quick verification with the example from the homework
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print("A:\n", A)
print("ifftshift(A):\n", np.fft.ifftshift(A))
print("fftshift(A):\n", np.fft.fftshift(A))

### Preparation 2 & 3: Display Helper Functions
- `imshow_gray`: displays a grayscale image (like MATLAB's `imshow(im,[])`)
- `imshow_kspace`: displays k-space magnitude spectrum using `log(|d|+1)` scaling

In [ ]:
# --- Preparation 2 & 3: Display helper functions ---

def imshow_gray(im, title="", vrange=None, figsize=(6, 5)):
    """
    Display a grayscale image.
    vrange: [vmin, vmax] for display range.
    If None, uses [min, max] of im (like MATLAB's imshow(im,[])).
    """
    plt.figure(figsize=figsize)
    if vrange is not None:
        plt.imshow(im, cmap='gray', vmin=vrange[0], vmax=vrange[1])
    else:
        plt.imshow(im, cmap='gray')
    plt.title(title)
    plt.colorbar(shrink=0.8)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def imshow_kspace(d, title="", figsize=(6, 5)):
    """
    Display k-space magnitude spectrum using log(|d|+1) scaling.
    Equivalent to MATLAB: imshow(log(abs(d)+1),[])
    """
    mag = np.log(np.abs(d) + 1)
    plt.figure(figsize=figsize)
    plt.imshow(mag, cmap='gray')
    plt.title(title)
    plt.colorbar(shrink=0.8)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

print("Display helper functions defined.")

### Preparation 4: IQA Helper Function
Wrapper for PSNR and SSIM computation. Both images are normalized to the same scale before comparison.

In [ ]:
# --- Preparation 4: IQA metrics ---

def compute_iqa(img, ref):
    """
    Compute PSNR and SSIM between img and ref.
    Both images are normalized by their own max before comparison.
    """
    img_n = img / np.max(img)
    ref_n = ref / np.max(ref)
    p = compute_psnr(ref_n, img_n, data_range=1.0)
    s = compute_ssim(ref_n, img_n, data_range=1.0)
    print(f"  PSNR = {p:.2f} dB")
    print(f"  SSIM = {s:.4f}")
    return p, s

print("IQA helper function defined.")

### Preparation 5: Load GE Phantom Data

In [ ]:
# --- Preparation 5: Load GE phantom k-space data ---
data = loadmat('ge_phantom.mat')
print("Keys in ge_phantom.mat:", [k for k in data.keys() if not k.startswith('__')])

# The variable name may vary; inspect and assign
for k in data.keys():
    if not k.startswith('__'):
        print(f"  '{k}' -> shape: {data[k].shape}, dtype: {data[k].dtype}")

---
## Part 1: Image Domain and Fourier Domain for Photos (20 pts)

### Part 1.1: Load Photo, Convert to Grayscale, Display Image and Magnitude Spectrum

**Instructions:** Place a photo file (e.g., `photo.jpg`) in the same folder as this notebook, then update the filename below.

In [ ]:
# --- Part 1.1: Read photo, convert to grayscale, display image & spectrum ---

# UPDATE the filename to your photo
photo = imread('photo.jpg')  # <-- change filename as needed

# Convert to grayscale (float64)
if photo.ndim == 3:
    im_ref = rgb2gray(photo).astype(np.float64)  # values in [0, 1]
else:
    im_ref = photo.astype(np.float64)

Nx, Ny = im_ref.shape
print(f"Reference image size: Nx={Nx}, Ny={Ny}")

# Display the reference image
imshow_gray(im_ref, title='Part 1.1: Reference Image $m_{ref}$')

# Centered 2D FFT and display magnitude spectrum
D_ref = fft2c(im_ref)
imshow_kspace(D_ref, title='Part 1.1: Magnitude Spectrum of $m_{ref}$')

### Part 1.2: Downsampled Image (by 2)
Keep every other pixel in both x- and y-directions.

In [ ]:
# --- Part 1.2: Downsample by 2 in both directions ---

m_down2 = im_ref[::2, ::2]  # keep every other pixel
print(f"Downsampled image size: {m_down2.shape}")

# Display downsampled image
imshow_gray(m_down2, title='Part 1.2: Downsampled Image (by 2)')

# Display its magnitude spectrum
D_down2 = fft2c(m_down2)
imshow_kspace(D_down2, title='Part 1.2: Magnitude Spectrum (downsampled by 2)')

### Part 1.3: Zero Padding in Fourier Domain (downsample by 2)
Pad the spectrum of the downsampled image with zeros to restore the original size $N_x \times N_y$.

In [ ]:
# --- Part 1.3: Zero-pad spectrum of 2x-downsampled image back to Nx x Ny ---

# Create zero array of original size
D_zp2 = np.zeros((Nx, Ny), dtype=complex)

# Place the downsampled spectrum at the center
Nx2, Ny2 = m_down2.shape
start_x = (Nx - Nx2) // 2
start_y = (Ny - Ny2) // 2
D_zp2[start_x:start_x+Nx2, start_y:start_y+Ny2] = D_down2

# Display zero-padded magnitude spectrum
imshow_kspace(D_zp2, title='Part 1.3: Zero-Padded Magnitude Spectrum (2x)')

# Inverse FFT to get the image (take magnitude to remove residual complex values)
m_zp2 = np.abs(ifft2c(D_zp2))

# Display reconstructed image
imshow_gray(m_zp2, title='Part 1.3: Zero-Padded Reconstruction (2x)')

# Normalize each image by its own max for error computation and IQA
m_zp2_n = m_zp2 / np.max(m_zp2)
im_ref_n = im_ref / np.max(im_ref)

# Error image
err_zp2 = np.abs(m_zp2_n - im_ref_n)
imshow_gray(err_zp2, title='Part 1.3: Error Image (2x zero-padded vs ref)')

# PSNR and SSIM
print("Part 1.3 IQA (2x downsampled + zero-padded vs reference):")
psnr_13, ssim_13 = compute_iqa(m_zp2, im_ref)

### Part 1.4: Downsample by 4 + Zero Pad
Repeat Part 1.3 with downsampling factor of 4.

In [ ]:
# --- Part 1.4: Downsample by 4 + zero-pad ---

m_down4 = im_ref[::4, ::4]
print(f"Downsampled (by 4) image size: {m_down4.shape}")

# Display downsampled image
imshow_gray(m_down4, title='Part 1.4: Downsampled Image (by 4)')

# FFT of downsampled image
D_down4 = fft2c(m_down4)
imshow_kspace(D_down4, title='Part 1.4: Magnitude Spectrum (downsampled by 4)')

# Zero-pad spectrum to Nx x Ny
D_zp4 = np.zeros((Nx, Ny), dtype=complex)
Nx4, Ny4 = m_down4.shape
start_x4 = (Nx - Nx4) // 2
start_y4 = (Ny - Ny4) // 2
D_zp4[start_x4:start_x4+Nx4, start_y4:start_y4+Ny4] = D_down4

# Display zero-padded magnitude spectrum
imshow_kspace(D_zp4, title='Part 1.4: Zero-Padded Magnitude Spectrum (4x)')

# Inverse FFT (take magnitude)
m_zp4 = np.abs(ifft2c(D_zp4))

# Display reconstructed image
imshow_gray(m_zp4, title='Part 1.4: Zero-Padded Reconstruction (4x)')

# Error image (normalize each by own max)
m_zp4_n = m_zp4 / np.max(m_zp4)
err_zp4 = np.abs(m_zp4_n - im_ref_n)
imshow_gray(err_zp4, title='Part 1.4: Error Image (4x zero-padded vs ref)')

# PSNR and SSIM
print("Part 1.4 IQA (4x downsampled + zero-padded vs reference):")
psnr_14, ssim_14 = compute_iqa(m_zp4, im_ref)

### Part 1.5: Comments on PSNR and SSIM

**Your comments here:**

- The 4x downsampled + zero-padded image has visibly lower quality (more blurring, loss of fine detail) compared to the 2x case.
- PSNR and SSIM both decrease with higher downsampling factors, which is consistent with the visual observation that more information is lost.
- Zero-padding in the Fourier domain effectively interpolates the image (sinc interpolation), preserving the low-frequency content but not recovering the lost high-frequency details.
- Both PSNR and SSIM accurately reflect the visual degradation: lower values correspond to visibly worse images.

*(Adjust these comments based on your actual numerical results.)*